In [2]:
!pip install transformers datasets scikit-learn

In [3]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import accuracy_score, f1_score

In [4]:
# load data set
dataset = load_dataset("ag_news")

train_data = dataset["train"].shuffle(seed=42).select(range(2000))
test_data  = dataset["test"].shuffle(seed=42).select(range(500))

label_names = ["World", "Sports", "Business", "Sci/Tech"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
# naive baseline
# keywoard rules

# rules
SPORTS = ["match", "game", "league", "goal", "season"]
BUSINESS = ["stock", "market", "company", "profit", "bank", "shares"]
TECH = ["technology", "software", "computer", "ai", "internet"]

# baseline predictor
def baseline_predict(text):
    text = text.lower()
    if any(k in text for k in SPORTS):
        return 1  # Sports
    elif any(k in text for k in BUSINESS):
        return 2  # Business
    elif any(k in text for k in TECH):
        return 3  # Sci/Tech
    else:
        return 0  # World

# execute baseline predictor
y_true = []
y_pred = []

for example in test_data:
    y_true.append(example["label"])
    y_pred.append(baseline_predict(example["text"]))

baseline_acc = accuracy_score(y_true, y_pred)
baseline_f1  = f1_score(y_true, y_pred, average="macro")

print("Baseline Accuracy:", baseline_acc)
print("Baseline F1:", baseline_f1)

Baseline Accuracy: 0.38
Baseline F1: 0.40337445592104604


In [7]:
# AI pipeline

# load model & tokenizer
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4
)

# tokenization
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_data = train_data.map(tokenize, batched=True)
test_data  = test_data.map(tokenize, batched=True)

train_data.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_data.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# training setup
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro")
    }

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    logging_steps=50,
    save_strategy="no",
    report_to="none"
)

# trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# execute
trainer.train()
bert_results = trainer.evaluate()

print("BERT Results:", bert_results)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

/tmp/ipython-input-4047717298.py:47: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
50,0.740400
100,0.420400
150,0.310000
200,0.239100
250,0.244200


BERT Results: {'eval_loss': 0.40637049078941345, 'eval_accuracy': 0.878, 'eval_f1': 0.8791309712306288, 'eval_runtime': 2.0069, 'eval_samples_per_second': 249.14, 'eval_steps_per_second': 15.945, 'epoch': 2.0}


In [8]:
# compare
print(f"Baseline  | Accuracy: {baseline_acc:.3f}, F1: {baseline_f1:.3f}")
print(f"DistilBERT | Accuracy: {bert_results['eval_accuracy']:.3f}, "
      f"F1: {bert_results['eval_f1']:.3f}")

==== Final Comparison ====
Baseline  | Accuracy: 0.380, F1: 0.403
DistilBERT | Accuracy: 0.878, F1: 0.879


In [10]:
test_data = test_data.with_format(None)
predictions = trainer.predict(test_data)
bert_preds = predictions.predictions.argmax(axis=1)

count = 0
for i, example in enumerate(test_data):
    base = baseline_predict(example["text"])
    bert = bert_preds[i]
    true = example["label"]

    if base != true and bert == true:
        print("\nHeadline:", example["text"])
        print("True Label:", label_names[true])
        print("Baseline:", label_names[base])
        print("BERT:", label_names[bert])
        count += 1

    if count == 3:
        break


Headline: Indian board plans own telecast of Australia series The Indian cricket board said on Wednesday it was making arrangements on its own to broadcast next month #39;s test series against Australia, which is under threat because of a raging TV rights dispute.
True Label: Sports
Baseline: Sci/Tech
BERT: Sports

Headline: Nuggets 112, Raptors 106 Carmelo Anthony scored 30 points and Kenyon Martin added 24 points and 16 rebounds, helping the Denver Nuggets hold off the Toronto Raptors 112-106 Wednesday night.
True Label: Sports
Baseline: World
BERT: Sports

Headline: REVIEW: 'Half-Life 2' a Tech Masterpiece (AP) AP - It's been six years since Valve Corp. perfected the first-person shooter with "Half-Life." Video games have come a long way since, with better graphics and more options than ever. Still, relatively few games have mustered this one's memorable characters and original science fiction story.
True Label: Sci/Tech
Baseline: Sports
BERT: Sci/Tech
